# 04d — What a run costs, judge included

The run record's `cost_usd` (`study_case_executions`, summed onto `study_runs`)
is the **agent plus the simulated member**. The judge's tokens are counted per
call but never persisted, so its share is **reconstructed** here for the two
simulation runs: the baseline (`1a872c09`, D96) and the regression run
(`e097f78e`).

**How.** Every judge prompt is rebuilt exactly as `src/judge.py` renders it (the
case's criteria, the fixture summary, the execution's own day, the reply or the
two replies) and its input tokens are counted with Vertex `count_tokens`. Nothing
is generated and nothing is written. Output tokens are counted from the stored
rationales in the judge's response JSON (`study_scores`; both orderings of
`study_comparisons`). Priced with `src/pricing.py`, the same list rates as the
application's usage log.

**Google Places** is read, not estimated: the API logs every billable Google
call to `api_usage_events` with its per-call list price (`apps/api/src/lib/pricing.ts`)
and the trip it served, and each repetition has its own trip. Section 3 keeps
the agent's own calls apart from the ones the harness causes by building the
fixture trip.

**The model figures are a lower bound.** All three parties' logged output
excludes thinking tokens (`candidates_token_count`), which Gemini bills as
output.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import json
import pandas as pd

from src import cost_estimate
from src.cases import CASES_DIR, load_cases

pd.set_option("display.width", 200)

# --- parameters -------------------------------------------------------------
BASELINE_RUN = "1a872c09-7070-4789-a806-904969f9185d"    # B10, adopted (D96)
REGRESSION_RUN = "e097f78e-3326-4aea-acaa-0b904d3b0a1c"  # B11
# ---------------------------------------------------------------------------

baseline_cases = json.loads((pathlib.Path(CASES_DIR) / "baseline.json").read_text(encoding="utf-8"))["cases"]
count = cost_estimate.token_counter()
runs = {
    "baseline": cost_estimate.run_costs(BASELINE_RUN, count=count),
    "regression": cost_estimate.run_costs(REGRESSION_RUN, baseline_cases=baseline_cases, count=count),
}
print(f"suite: {len(load_cases())} cases; pricing version {runs['baseline'].attrs['pricing_version']}")
for name, f in runs.items():
    print(f"{name}: {len(f)} executions over {f['case'].nunique()} cases")

suite: 31 cases; pricing version 2026-07
baseline: 176 executions over 31 cases
regression: 176 executions over 31 cases


## 1. Per execution and per run

One execution is one repetition of one case. The baseline run is judged
point-wise only (one call per execution); the regression run is judged
point-wise **and** pair-wise against the adopted baseline entry, in both
orderings (two calls, D23).

In [2]:
def summary(f):
    judged = f[f["pointwise_in"] > 0]
    compared = f[f["pairwise_in"] > 0]
    return {
        "executions": len(f),
        "simulation, mean $/execution": round(f["simulation_usd"].mean(), 4),
        "point-wise judge, mean $/execution": round(f["pointwise_usd"].mean(), 4),
        "  tokens in / out (mean, judged)": f"{judged['pointwise_in'].mean():.0f} / {judged['pointwise_out'].mean():.0f}",
        "pair-wise judge, mean $/execution": round(f["pairwise_usd"].mean(), 4),
        "  tokens in / out (mean, compared)": (f"{compared['pairwise_in'].mean():.0f} / {compared['pairwise_out'].mean():.0f}"
                                              if len(compared) else "-"),
        "models total, mean $/execution": round(f["total_usd"].mean(), 4),
        "Places, agent, mean $/execution": round(f["agent_places_usd"].mean(), 4),
        "Places, fixture build, mean $/execution": round(f["fixture_places_usd"].mean(), 4),
        "judge share of models total": f"{f['judge_usd'].sum() / f['total_usd'].sum():.1%}",
        "simulation, run $": round(f["simulation_usd"].sum(), 2),
        "judge, run $": round(f["judge_usd"].sum(), 2),
        "models total, run $": round(f["total_usd"].sum(), 2),
        "Places, agent, run $": round(f["agent_places_usd"].sum(), 2),
        "Places, fixture build, run $": round(f["fixture_places_usd"].sum(), 2),
        "all in, run $": round((f["total_usd"] + f["agent_places_usd"] + f["fixture_places_usd"]).sum(), 2),
    }

pd.DataFrame({name: summary(f) for name, f in runs.items()})

,baseline,regression
executions,176,176
"simulation, mean $/execution",0.0257,0.0251
"point-wise judge, mean $/execution",0.0028,0.0028
"tokens in / out (mean, judged)",1385 / 76,1383 / 76
"pair-wise judge, mean $/execution",0.0,0.0063
"tokens in / out (mean, compared)",-,3343 / 145
"models total, mean $/execution",0.0284,0.0341
"Places, agent, mean $/execution",0.0343,0.0352
"Places, fixture build, mean $/execution",0.2,0.2
judge share of models total,9.7%,26.6%


## 2. Per case function

Where the cost sits. The spread is narrow: a two-turn scheduling execution costs
about a third more than a one-turn factual question, because the agent's cost
grows with turns but most executions end on a short second turn. On the judge
side the factual cases cost most, because their prompts carry reference data
(climate figures, forecasts) as well as the reply.

In [3]:
pooled = pd.concat([f.assign(run=n) for n, f in runs.items()], ignore_index=True)
pooled["function"] = pooled["case"].map({c.case_id: c.function for c in load_cases()})
(pooled.groupby(["function", "run"])[["simulation_usd", "judge_usd", "total_usd"]]
       .mean().round(4).unstack("run"))

simulation_usd            judge_usd            total_usd           
run                baseline regression  baseline regression  baseline regression
function                                                                        
advisory             0.0291     0.0293    0.0020     0.0071    0.0311     0.0364
factual_qa           0.0218     0.0192    0.0046     0.0144    0.0264     0.0336
scheduling           0.0321     0.0320    0.0018     0.0060    0.0339     0.0380
venue_lookup         0.0192     0.0204    0.0020     0.0069    0.0212     0.0274

## 3. Google Places

Read from `api_usage_events`, attributed to an execution by its trip.

- **Agent**: calls made during the agent's turns: `search_places` and the venue
  lookups `propose_plan` runs to locate drafted stops (both Text Search).
- **Fixture build**: creating each of the fixture's five blocks runs a Nearby
  Search to resolve its place, so every repetition pays for five before the
  member says anything. That is the harness's cost, identical in every
  execution, and says nothing about the agent.

Prices are the application's list estimates per call (Enterprise tier, because
the field masks request ratings and summaries). Google's monthly free usage
per SKU is not netted out, so the amount actually billed can be lower.

In [4]:
places = (pooled.groupby(["function", "run"])
               .agg(agent_calls=("agent_places_calls", "mean"), agent_usd=("agent_places_usd", "mean"),
                    fixture_calls=("fixture_places_calls", "mean"), fixture_usd=("fixture_places_usd", "mean"))
               .round(3).unstack("run"))
for name, f in runs.items():
    print(f"{name}: agent {int(f['agent_places_calls'].sum())} calls, ${f['agent_places_usd'].sum():.2f}; "
          f"fixture build {int(f['fixture_places_calls'].sum())} calls, ${f['fixture_places_usd'].sum():.2f}")
places

baseline: agent 151 calls, $6.04; fixture build 880 calls, $35.20
regression: agent 155 calls, $6.20; fixture build 880 calls, $35.20


agent_calls            agent_usd            fixture_calls            fixture_usd           
run             baseline regression  baseline regression      baseline regression    baseline regression
function                                                                                                
advisory           0.000      0.031     0.000      0.001           5.0        5.0         0.2        0.2
factual_qa         0.036      0.000     0.001      0.000           5.0        5.0         0.2        0.2
scheduling         1.615      1.596     0.065      0.064           5.0        5.0         0.2        0.2
venue_lookup       1.806      1.972     0.072      0.079           5.0        5.0         0.2        0.2

## 4. For the report

The sentence the baseline subchapter can carry, computed rather than typed.

In [5]:
b = runs["baseline"]
places_agent = b["agent_places_usd"].mean()
print(f"One execution of a test case cost about USD {b['total_usd'].mean():.3f} in model calls on average "
      f"(simulation USD {b['simulation_usd'].mean():.3f}, point-wise judging USD {b['pointwise_usd'].mean():.3f}), "
      f"so the baseline run of {len(b)} executions cost about USD {b['total_usd'].sum():.2f}, estimated from token "
      f"counts at list price and excluding the models' internal reasoning tokens. The agent's Google Places "
      f"lookups added USD {places_agent:.3f} per execution (USD {b['agent_places_usd'].sum():.2f}), and building "
      f"the fixture trip for each repetition USD {b['fixture_places_usd'].mean():.2f} "
      f"(USD {b['fixture_places_usd'].sum():.2f}), at list price before Google's free monthly allowance.")

One execution of a test case cost about USD 0.028 in model calls on average (simulation USD 0.026, point-wise judging USD 0.003), so the baseline run of 176 executions cost about USD 5.00, estimated from token counts at list price and excluding the models' internal reasoning tokens. The agent's Google Places lookups added USD 0.034 per execution (USD 6.04), and building the fixture trip for each repetition USD 0.20 (USD 35.20), at list price before Google's free monthly allowance.
